In [1]:
#imports
import os
import numpy as np
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt

In [4]:
#Class for loading images and depth maps

class NYUDataset(Dataset):
    def __init__(self, img_dir, depth_dir):
        self.img_dir = img_dir
        self.depth_dir = depth_dir
        self.files = sorted(os.listdir(img_dir))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_name = self.files[idx]

        img_path = os.path.join(self.img_dir, img_name)
        depth_path = os.path.join(self.depth_dir, img_name)

        #Loading image
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

In [44]:
#Class for loading images and depth maps

class NYUDataset(Dataset):
    def __init__(self, img_dir, depth_dir):
        self.img_dir = img_dir
        self.depth_dir = depth_dir
        self.files = sorted(os.listdir(img_dir))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_name = self.files[idx]

        img_path = os.path.join(self.img_dir, img_name)
        #depth_path = os.path.join(self.depth_dir, img_name)

        #changing extension to .png
        base_name = os.path.splitext(img_name)[0]
        depth_name = base_name + ".png"
        depth_path = os.path.join(self.depth_dir, depth_name)
        
        #Loading image
        image = cv2.imread(img_path)
        if image is None:
            print(f"Image not found: {img_path}")
            return None
            
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (256, 256))
        image = image / 255.0

        # Load depth
        depth = cv2.imread(depth_path, cv2.IMREAD_UNCHANGED)
        if depth is None:
            print(f"Depth not found: {depth_path}")
            return None

        depth = cv2.resize(depth, (256, 256))
        depth = depth.astype(np.float32) / 1000.0

        # Convert to tensor
        image = torch.tensor(image).permute(2, 0, 1).float()
        depth = torch.tensor(depth).unsqueeze(0).float()

        return image, depth

what the above does:
resize = uniform input size
normalize = stable training
tensor = PyTorch format

In [49]:
#DataLoader

train_dataset = NYUDataset(
    "dataset/nyu_train_preprocessed/images",
    "dataset/nyu_train_preprocessed/depth"
)

train_dataset.files = train_dataset.files[:500]

def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    return torch.utils.data.dataloader.default_collate(batch)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn, num_workers=0)

In [50]:
#Our model

class DepthModel(nn.Module):
    def __init__(self):
        super(DepthModel, self).__init__()

        #Encoder
        self.enc = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        #Decoder
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 2, stride=2),
            nn.ReLU(),

            nn.ConvTranspose2d(64, 32, 2, stride=2),
            nn.ReLU(),

            nn.ConvTranspose2d(32, 1, 2, stride=2)
        )

    def forward(self, x):
        x = self.enc(x)
        x = self.dec(x)
        return x

what the above does:
RGB Image -> Encoder -> compressed features -> decoder -> depth map

In [51]:
#Setup training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DepthModel().to(device)

criterion = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [23]:
#checking which files are not present in depth but are present in rgb

missing = 0

for file in os.listdir("dataset/nyu_train_preprocessed/images"):
    base = os.path.splitext(file)[0]
    depth_file = base + ".png"
    
    if not os.path.exists(f"dataset/nyu_train_preprocessed/depth/{depth_file}"):
        print("Missing depth for:", depth_file)
        missing += 1

print("Total missing:", missing)

Total missing: 0


In [53]:
#Training Loop

epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for i, (images, depths) in enumerate(train_loader):
        print(f"Epoch {epoch+1}, Batch {i}")
        images = images.to(device)
        depths = depths.to(device)

        preds = model(images)

        loss = criterion(preds, depths)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if i == 20:
            break
    print(f"Epoch {epoch+1}, Loss: {total_loss/(i+1)}")

Epoch 1, Batch 0
Epoch 1, Batch 1
Epoch 1, Batch 2
Epoch 1, Batch 3
Epoch 1, Batch 4
Epoch 1, Batch 5
Epoch 1, Batch 6
Epoch 1, Batch 7
Epoch 1, Batch 8
Epoch 1, Batch 9
Epoch 1, Batch 10
Epoch 1, Batch 11
Epoch 1, Batch 12
Epoch 1, Batch 13
Epoch 1, Batch 14
Epoch 1, Batch 15
Epoch 1, Batch 16
Epoch 1, Batch 17
Epoch 1, Batch 18
Epoch 1, Batch 19
Epoch 1, Batch 20
Epoch 1, Loss: 0.17441558908848537
Epoch 2, Batch 0
Epoch 2, Batch 1
Epoch 2, Batch 2
Epoch 2, Batch 3
Epoch 2, Batch 4
Epoch 2, Batch 5
Epoch 2, Batch 6
Epoch 2, Batch 7
Epoch 2, Batch 8
Epoch 2, Batch 9
Epoch 2, Batch 10
Epoch 2, Batch 11
Epoch 2, Batch 12
Epoch 2, Batch 13
Epoch 2, Batch 14
Epoch 2, Batch 15
Epoch 2, Batch 16
Epoch 2, Batch 17
Epoch 2, Batch 18
Epoch 2, Batch 19
Epoch 2, Batch 20
Epoch 2, Loss: 0.11992508350383668
Epoch 3, Batch 0
Epoch 3, Batch 1
Epoch 3, Batch 2
Epoch 3, Batch 3
Epoch 3, Batch 4
Epoch 3, Batch 5
Epoch 3, Batch 6
Epoch 3, Batch 7
Epoch 3, Batch 8
Epoch 3, Batch 9
Epoch 3, Batch 10
Epoch 

In [41]:
print(device)

cpu
